In [ ]:
from ase.io import read
import numpy as np
from ase.neighborlist import NeighborList

windows = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 15, 20, 50, 100, 200]

atoms = []
for window in windows:
    atoms.append(read(f'../data/Time-averaging/1000K_smooth_{window}_selected.xyz', ':'))

In [ ]:
def find_BX_vectors(atom, B_indices, X_indices, cutoff):
    
    cutoffs = [cutoff/2] * len(atom)
    nl = NeighborList(cutoffs, self_interaction=False, bothways=True, skin=0.0)
    nl.update(atom)

    BX_vectors = []
    for B_index in B_indices:
        index, offsets = nl.get_neighbors(B_index)
        BX_bonds = []
        for X_index, offset in zip(index, offsets):
            if X_index in X_indices:
                vec = atom.positions[X_index] + np.dot(offset, atom.get_cell()) - atom.positions[B_index]
                BX_bonds.append([vec, X_index])
        BX_vectors.append(BX_bonds)

    return BX_vectors

In [ ]:
def find_long_bonds(atom, B_indices, X_indices, cutoff, long_cutoff):
    
    BX_vectors = find_BX_vectors(atom, B_indices, X_indices, cutoff)
    JT_array = np.zeros(len(atom), dtype=int)
    long_array = np.zeros(len(atom), dtype=int)

    for B_index, BX_bonds in zip(B_indices, BX_vectors):
        long_bonds = []
        for BX_bond in BX_bonds:
            if np.linalg.norm(BX_bond[0]) > long_cutoff:
                long_array[B_index] = 1
                long_array[BX_bond[1]] = 1
                long_bonds.append(BX_bond[0]/np.linalg.norm(BX_bond[0]))
                
        if len(long_bonds) == 2:
            if np.abs(np.dot(long_bonds[0], long_bonds[1])) < 0.35:
                JT_array[B_index] = 3
            elif np.abs(np.dot(long_bonds[0], long_bonds[1])) > 0.7:
                JT_array[B_index] = 2
            else:
                print('B_index:', B_index, 'long_bonds:', long_bonds, 'dot:', np.dot(long_bonds[0], long_bonds[1]))
        else:
            JT_array[B_index] = 1

    atom.arrays['long'] = long_array
    atom.arrays['JT'] = JT_array

    return atom


In [ ]:
B_site = 'Mn'
X_site = 'O'

B_indices = [site.index for site in atoms[0][0] if site.symbol == B_site]
X_indices = [site.index for site in atoms[0][0] if site.symbol == X_site]

cutoff = 3.0

long_cutoff = 2.05

coded_atoms = []

for smooth in atoms:
    coded_atoms.append([find_long_bonds(atom, B_indices, X_indices, cutoff, long_cutoff) for atom in smooth])

In [ ]:
counts_1_mean = []
counts_2_mean = []
counts_3_mean = []

counts_1_std = []
counts_2_std = []
counts_3_std = []

for smooth in atoms:

    counts_1 = [np.count_nonzero(array == 1) for atom in smooth for array in [atom.arrays["JT"]]]
    counts_2 = [np.count_nonzero(array == 2) for atom in smooth for array in [atom.arrays["JT"]]]
    counts_3 = [np.count_nonzero(array == 3) for atom in smooth for array in [atom.arrays["JT"]]]

    counts_1_mean.append(np.mean(counts_1))
    counts_2_mean.append(np.mean(counts_2))
    counts_3_mean.append(np.mean(counts_3))

    counts_1_std.append(np.std(counts_1))
    counts_2_std.append(np.std(counts_2))
    counts_3_std.append(np.std(counts_3))
    

In [ ]:
percentage_mean_1 = [i*100/216 for i in counts_1_mean]
percentage_mean_2 = [i*100/216 for i in counts_2_mean]
percentage_mean_3 = [i*100/216 for i in counts_3_mean]

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()

time = [i*0.01 for i in windows]

ax.errorbar(time, percentage_mean_1, yerr=counts_1_std, label="Others", color="#DDDDDD", fmt='o-', markersize=5)
ax.errorbar(time, percentage_mean_2, yerr=counts_2_std, label="X/Y/Z-axis", color="#33BBEE", fmt='o-', markersize=5)
ax.errorbar(time, percentage_mean_3, yerr=counts_3_std, label="L-type", color="#EE3377", fmt='o-', markersize=5)

ax.set_xlabel("Timescale (ps)")
ax.set_ylabel("Percentage of octahedra (%)")
ax.set_xscale('log')

plt.legend(frameon=False) 
plt.savefig('JT_percentage_1000K.png', bbox_inches='tight')  
plt.show()